In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ── Model v2 – no nn.Sequential, 3 conv layers ────────────────────────────────
class SkinConvNetV1(nn.Module):
    """3-layer ConvNet written with explicit layer attributes (no nn.Sequential)."""

    def __init__(self, num_classes: int = 4):
        super().__init__()

        # ── Conv blocks ──────────────────────────────────────────────────────
        # Block 1 – 3 → 32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)

        # Block 2 – 32 → 64
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)

        # Block 3 – 64 → 128
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)

        self.pool    = nn.MaxPool2d(2)
        self.avgpool = nn.AdaptiveAvgPool2d(4)
        self.relu    = nn.ReLU(inplace=True)

        # ── Classifier ───────────────────────────────────────────────────────
        self.flatten = nn.Flatten()
        self.fc1     = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2     = nn.Linear(256, num_classes)

    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1(self.conv1(x)))   # 3×128×128  → 32×128×128
        x = self.pool(x)                          # → 32×64×64

        # Block 2
        x = self.relu(self.bn2(self.conv2(x)))   # → 64×64×64
        x = self.pool(x)                          # → 64×32×32

        # Block 3
        x = self.relu(self.bn3(self.conv3(x)))   # → 128×32×32
        x = self.avgpool(x)                       # → 128×4×4

        # Classifier
        x = self.flatten(x)                       # → 2048
        x = self.relu(self.fc1(x))               # → 256
        x = self.dropout(x)
        x = self.fc2(x)                           # → num_classes
        return x




In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
class SkinConvNetV2(nn.Module):
    """Simple 4-layer ConvNet for skin-lesion classification."""

    def __init__(self, num_classes: int = 4):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 – 3 → 32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),   # 128×128
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # 64×64

            # Block 2 – 32 → 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 64×64
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # 32×32

            # Block 3 – 64 → 128
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 32×32
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                               # 16×16

            # Block 4 – 128 → 256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),# 16×16
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),                       # 4×4 (fixed output size)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



In [ ]:
from torchvision import models

# ── ResNet-based model ─────────────────────────────────────────────────────────
class SkinLesionResNet(nn.Module):
    """Fine-tuned ResNet-18 for skin-lesion classification."""

    def __init__(self, num_classes: int = 4):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT 
        backbone = models.resnet18(weights=weights)

        # Replace the final FC layer to match num_classes
        in_features = backbone.fc.in_features          # 512 for ResNet-18
        backbone.fc = nn.Linear(in_features, num_classes)

        self.model = backbone

    def forward(self, x):
        return self.model(x)



In [ ]:
import os
import random
from glob import glob
from collections import defaultdict
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ── Custom Dataset with built-in split ────────────────────────────────────────
class SkinLesionDataset(Dataset):
    """
    Loads all images from a folder tree, performs a stratified
    train / val / test split inside the constructor.

    folder structure:
        root/
          class_A/img1.jpg ...
          class_B/img2.jpg ...

    Usage:
        train_ds = SkinLesionDataset(root, split="train")
        val_ds   = SkinLesionDataset(root, split="val")
        test_ds  = SkinLesionDataset(root, split="test")
    """

    _IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp")

    def __init__(
        self,
        root: str,
        split: str = "train",       # "train" | "val" | "test"
        train_ratio: float = 0.70,
        val_ratio:   float = 0.15,  # test_ratio = 1 - train_ratio - val_ratio
        transform=None,
        seed: int = 42,
    ):
        assert split in ("train", "val", "test"), "split must be 'train', 'val', or 'test'"


        paths = glob(os.path.join(root,"*","*"))
        label_names = [os.path.basename(os.path.dirname(p)) for p in paths]
        labels = {cls: idx for idx, cls in enumerate(sorted(set(labels)))}
        self.transform = transform

        # split the dataset
        random.seed(seed)
        data = list(zip(paths, label_names, labels))
        random.shuffle(data)
        self.paths, labels = zip(*data)
        num_total = len(self.paths)
        num_train = int(train_ratio * num_total)
        num_val   = int(val_ratio * num_total)  


        if split == "train":
            self.file_paths = self.paths[:num_train]
            self.labels     = [self.labelsidx[lbl] for lbl in labels[:num_train]]
        elif split == "val":
            self.file_paths = self.paths[num_train:num_train+num_val]
            self.labels     = [self.labelsidx[lbl] for lbl in labels[num_train:num_train+num_val]]
        else:  # test
            self.file_paths = self.paths[num_train+num_val:]
            self.labels     = [self.labelsidx[lbl] for lbl in labels[num_train+num_val:]]



    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img   = Image.open(self.file_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label





In [ ]:
train_ds = SkinLesionDataset(root="data/skin_lesions", split="train")
val_ds   = SkinLesionDataset(root="data/skin_lesions", split="val"  )
test_ds  = SkinLesionDataset(root="data/skin_lesions", split="test")


batch_size = 10
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)



In [ ]:
model = SkinConvNetV2(num_classes=4)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


for epoch in range(1, 11):
    model.train()
    total_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)

        
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item() * imgs.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch:02d} - Train Loss: {avg_loss:.4f}")